In [13]:
import sys
import os
from os.path import join
import glob
from copy import deepcopy
import logging
from tqdm import tqdm

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display, clear_output

sys.path.insert(0, "/home/2649/repos/ABC-SN/code")
import abcsn_config
import abcsn_training
import data_degrading as dg
import data_plotting as dplt
import data_preparation as dp
import preprocessing

sys.path.insert(0, "/home/2649/repos/spec_res/code")
import review_spectrum as rs
import spectral_features as sf
import measure_signal as ms

from icecream import ic
from importlib import reload

reload(rs)
reload(sf)

REPO_DIR = "/home/2649/repos/spec_res"
DATA_DIR = join(REPO_DIR, "data")
BIANCO_DENOISING_DIR = join(DATA_DIR, "original_resolution_parquet/bianco_denoising")

In [14]:
df_data, df_metadata, wvl = rs.load_sn_data()
FFTd_S_data, FFTd_S_metadata, FFTd_N_data, FFTd_N_metadata = rs.load_FFTdenoised_data()
# df_SNRmetadata = rs.create_SNRmetadata_file()

In [15]:
sort_cols = ["SN Subtype", "SN Name", "Spectral Phase"]
df_metadata_sorted = df_metadata.sort_values(sort_cols).copy(deep=True)

In [19]:
logging.getLogger().setLevel(logging.WARNING)
reload(ms)
specsnr_objs = []
for i in tqdm(range(df_data.shape[0])):
    sn_name = df_metadata_sorted["SN Name"].iloc[i]
    sn_type = df_metadata_sorted["SN Subtype"].iloc[i]
    sn_phase = df_metadata_sorted["Spectral Phase"].iloc[i]
    if sn_type != "Ia-norm":
        continue
    
    spectrum = rs.get_spectrum(df_data, df_metadata, sn_name, sn_phase)
    specsnr = ms.SpectrumSNR(
        sn_name,
        sn_type,
        sn_phase,
        wvl,
        spectrum)
    # specsnr.summarize()
    specsnr.set_spectral_feature()
    specsnr.denoise_gaussian(10)
    specsnr.find_spectral_line(feature_search_bounds=(500, 0), plot=False)
    specsnr.find_spectral_shoulders(plot=False)
    specsnr.calc_pEW(plot=False)
    specsnr.measure_feature_noise(plot=False)
    specsnr.measure_SNR(plot=False)
    specsnr_objs.append(specsnr)

  8%|▊         | 313/3764 [00:20<03:42, 15.48it/s]


AssertionError: More than one spectrum called 'sn1994D' at phase '49.5'.